# 01 — Task Environment

Implements and verifies the task environment from:
> Fang, Gao, Xia, Cheng et al. (2026) — *Imbalanced learning efficiency and cognitive effort in individuals with SUD*

All logic lives in `../env.py`. This notebook verifies it against the reward matrix in `model.md` and visualizes the tree.

In [ ]:
import sys
sys.path.insert(0, '..')   # so we can import from the project root

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from env import Env

## 1. Reward Matrix

Expected values from `model.md`:

| Room | d(s)      | A easy | B easy | A hard | B hard | Test |
|------|-----------|--------|--------|--------|--------|------|
| s=4  | [10,0,0]  | −10    | **10** | −20    | **10** | 10   |
| s=5  | [4,5,10]  | 1      | −1     | −3     | −6     | 19   |
| s=6  | [9,9,10]  | 0      | 0      | −9     | −9     | **28** |
| s=7  | [0,8,2]   | 8      | −8     | 8      | −16    | 10   |
| s=8  | [0,10,6]  | **10** | −10    | **10** | −20    | 16   |
| s=9  | [8,0,0]   | −8     | 8      | −16    | 8      | 8    |

In [ ]:
mat = Env.reward_matrix()

# Build DataFrame: rows = terminal rooms, columns = goals
goal_order = ['A_easy', 'B_easy', 'A_hard', 'B_hard', 'test']
df = pd.DataFrame(
    {g: [mat[g][s] for s in Env.TERMINAL] for g in goal_order},
    index=[f's={s}' for s in Env.TERMINAL]
)
df.index.name = 'Room'
print(df.to_string())

In [ ]:
# Cross-check: verify each cell against model.md ground truth
expected = {
    'A_easy': {4: -10, 5:  1, 6:  0, 7:  8, 8: 10, 9: -8},
    'B_easy': {4:  10, 5: -1, 6:  0, 7: -8, 8:-10, 9:  8},
    'A_hard': {4: -20, 5: -3, 6: -9, 7:  8, 8: 10, 9:-16},
    'B_hard': {4:  10, 5: -6, 6: -9, 7:-16, 8:-20, 9:  8},
    'test':   {4:  10, 5: 19, 6: 28, 7: 10, 8: 16, 9:  8},
}

all_ok = True
for goal, rooms in expected.items():
    for s, r_expected in rooms.items():
        r_got = mat[goal][s]
        if not np.isclose(r_got, r_expected):
            print(f'FAIL  goal={goal}  s={s}: expected {r_expected}, got {r_got}')
            all_ok = False

print('All reward values match model.md ✓' if all_ok else 'Some values mismatched — check env.py')

In [ ]:
# Optimal rooms per goal
for g in goal_order:
    best = Env.optimal_room(g)
    r    = mat[g][best]
    print(f'{g:10s}  →  optimal room s={best}  (R={r})')

## 2. Reward Heatmap

In [ ]:
z    = df.values
text = [[str(int(v)) for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z,
    x=goal_order,
    y=[f's={s}' for s in Env.TERMINAL],
    text=text,
    texttemplate='%{text}',
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='Reward'),
))

fig.update_layout(
    title='Reward matrix R = w_g · φ(s)',
    xaxis_title='Goal',
    yaxis_title='Terminal room',
    yaxis_autorange='reversed',
    width=550, height=400,
)
fig.show()

## 3. Tree Visualization

In [ ]:
# Node positions
pos = {
    0: ( 0.0,  2.0),   # root
    1: (-2.0,  1.0),   # intermediate left
    2: ( 0.0,  1.0),   # intermediate mid
    3: ( 2.0,  1.0),   # intermediate right
    4: (-2.75, 0.0),   # terminal
    5: (-1.25, 0.0),
    6: (-0.5,  0.0),
    7: ( 0.5,  0.0),
    8: ( 1.5,  0.0),
    9: ( 2.5,  0.0),
}

# Edges
edges = [(p, c) for p, children in Env.TRANSITIONS.items() for c in children.values()]

edge_x, edge_y = [], []
for (p, c) in edges:
    x0, y0 = pos[p]
    x1, y1 = pos[c]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

# Node colours: non-terminal = blue, terminal = coloured by max reward across training tasks
max_training_reward = [
    max(mat[g][s] for g in ['A_easy', 'B_easy', 'A_hard', 'B_hard'])
    if s in Env.TERMINAL else 0
    for s in Env.STATES
]

node_labels = [
    f's={s}<br>d={Env.PHI[s].astype(int).tolist()}' if s in Env.TERMINAL
    else f's={s}'
    for s in Env.STATES
]

node_x = [pos[s][0] for s in Env.STATES]
node_y = [pos[s][1] for s in Env.STATES]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(color='#aaa', width=1.5),
    hoverinfo='none',
))

fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    marker=dict(
        size=40,
        color=['#4a90d9' if s in Env.NON_TERMINAL else '#e07b39' for s in Env.STATES],
        line=dict(width=2, color='white'),
    ),
    text=[f's={s}' for s in Env.STATES],
    textposition='middle center',
    textfont=dict(color='white', size=13),
    hovertext=node_labels,
    hoverinfo='text',
))

# Action labels on edges
action_names_s0 = {0: 'left', 1: 'mid', 2: 'right'}
action_names    = {0: 'left', 1: 'right'}

for p, children in Env.TRANSITIONS.items():
    an = action_names_s0 if p == 0 else action_names
    for a, c in children.items():
        mx = (pos[p][0] + pos[c][0]) / 2
        my = (pos[p][1] + pos[c][1]) / 2
        fig.add_annotation(
            x=mx, y=my,
            text=an[a],
            showarrow=False,
            font=dict(size=10, color='#555'),
            bgcolor='white',
            opacity=0.8,
        )

fig.update_layout(
    title='Task tree (blue = non-terminal, orange = terminal)',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    showlegend=False,
    width=700, height=420,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.show()

## 4. Feature Vectors φ(s)

In [ ]:
phi_df = pd.DataFrame(
    {f's={s}': Env.PHI[s] for s in Env.TERMINAL},
    index=['resource_0', 'resource_1', 'resource_2']
).T
phi_df.index.name = 'Room'

fig = px.bar(
    phi_df.reset_index().melt(id_vars='Room', var_name='Resource', value_name='Quantity'),
    x='Room', y='Quantity', color='Resource', barmode='group',
    title='Feature vectors φ(s) = d(s) for terminal rooms',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(width=600, height=360)
fig.show()

## 5. Trial Sequence

In [ ]:
seq = Env.make_trial_sequence(seed=42)
print(f'Total trials: {len(seq)}')
print(f'Goal counts:  {dict(zip(*np.unique(seq, return_counts=True)))}')
print()
print('First 20 trials:', seq[:20])

In [ ]:
# Visualize goal sequence
goal_to_int = {g: i for i, g in enumerate(Env.TRAINING_GOALS)}
seq_int = [goal_to_int[g] for g in seq]

fig = go.Figure(go.Scatter(
    x=list(range(len(seq))),
    y=seq,
    mode='markers',
    marker=dict(
        color=seq_int,
        colorscale='Viridis',
        size=6,
    ),
))
fig.update_layout(
    title='Trial sequence (80 training trials)',
    xaxis_title='Trial',
    yaxis_title='Goal',
    width=700, height=300,
)
fig.show()

## 6. Transition sanity check

Walk the optimal path for each training goal and confirm the reward.

In [ ]:
# Optimal action sequences from model.md:
#   Type A: right (s=0→s=3) then left (s=3→s=8)
#   Type B: left  (s=0→s=1) then left (s=1→s=4)
optimal_paths = {
    'A_easy': [2, 0],   # right then left
    'A_hard': [2, 0],
    'B_easy': [0, 0],   # left then left
    'B_hard': [0, 0],
    'test':   [1, 0],   # mid then left → s=6
}

for goal, actions in optimal_paths.items():
    s = 0
    path = [s]
    for a in actions:
        s = Env.step(s, a)
        path.append(s)
    r = Env.reward(s, Env.GOALS[goal])
    best_s = Env.optimal_room(goal)
    status = '✓' if s == best_s else f'✗ (expected s={best_s})'
    print(f'{goal:10s}  path={path}  terminal=s={s}  R={r:5.0f}  {status}')